# Predictive Demand Forecasting Using Data Science

This notebook implements the complete predictive modeling workflow for the Data Science Capstone Project. It includes data loading, cleaning, exploratory analysis, time-series features, chronological train/test splitting, baseline forecasting, ARIMA, Random Forest regression, and model evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option('display.max_columns', None)

## 1. Load the Dataset

In [ ]:
df = pd.read_csv('../data/daily-demand-series.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())

## 2. Exploratory Data Analysis

In [ ]:
print(df.describe(include='all'))

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df['date'], df['demand'], marker='o')
plt.title('Daily Demand Trend')
plt.xlabel('Date')
plt.ylabel('Demand')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Stationarity Test

In [ ]:
adf_stat, p_value, _, _, critical_values, _ = adfuller(df['demand'])
print('ADF statistic:', round(adf_stat, 4))
print('p-value:', round(p_value, 4))
print('Critical values:', critical_values)
print('\nInterpretation:')
print('If p-value < 0.05, the series is considered stationary at the 5% significance level.')

## 4. Feature Engineering

In [ ]:
d = df.copy()
d['lag1'] = d['demand'].shift(1)
d['lag7'] = d['demand'].shift(7)
d['rolling7'] = d['demand'].shift(1).rolling(7).mean()
d_ml = d.dropna().reset_index(drop=True)
d_ml.head()

## 5. Chronological Train/Test Split

In [ ]:
# Reserve the final 6 observations as an unseen test set.
cutoff = df['date'].iloc[-7]
train = df[df['date'] < cutoff].copy()
test = df[df['date'] >= cutoff].copy()
print('Training observations:', len(train))
print('Test observations:', len(test))

## 6. Baseline Model — Previous-Day Demand

In [ ]:
baseline_pred = test['demand'].shift(1).copy()
baseline_pred.iloc[0] = train['demand'].iloc[-1]
baseline_pred = baseline_pred.to_numpy()

## 7. ARIMA Model

In [ ]:
arima = ARIMA(train['demand'], order=(1, 1, 1)).fit()
arima_pred = arima.forecast(steps=len(test))
print(arima.summary())

## 8. Random Forest Model

In [ ]:
features = ['lag1', 'lag7', 'rolling7', 'marketing_event', 'holiday']
train_ml = d_ml[d_ml['date'] < cutoff]
test_ml = d_ml[d_ml['date'] >= cutoff]

rf = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42)
rf.fit(train_ml[features], train_ml['demand'])
rf_pred = rf.predict(test_ml[features])

## 9. Model Evaluation

In [ ]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((np.array(y_true) - np.array(y_pred)) / np.array(y_true))) * 100
    return mae, rmse, mape

m_base = evaluate(test['demand'], baseline_pred)
m_arima = evaluate(test['demand'], arima_pred)
m_rf = evaluate(test_ml['demand'], rf_pred)

results = pd.DataFrame({
    'Model': ['Previous-Day Baseline', 'ARIMA(1,1,1)', 'Random Forest'],
    'MAE': [m_base[0], m_arima[0], m_rf[0]],
    'RMSE': [m_base[1], m_arima[1], m_rf[1]],
    'MAPE (%)': [m_base[2], m_arima[2], m_rf[2]]
})
results.round(2)

## 10. Actual vs Predicted Demand

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(test['date'], test['demand'], marker='o', label='Actual')
plt.plot(test['date'], baseline_pred, marker='o', label='Baseline')
plt.plot(test['date'], arima_pred, marker='o', label='ARIMA')
plt.plot(test_ml['date'], rf_pred, marker='o', label='Random Forest')
plt.title('Actual vs Predicted Demand')
plt.xlabel('Date')
plt.ylabel('Demand')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 11. Feature Importance

In [ ]:
importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importance)
importance.plot(kind='bar', figsize=(8,4), title='Random Forest Feature Importance')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

## 12. Conclusion

The workflow compares a simple baseline with ARIMA and Random Forest forecasting. The model with the lowest MAE on the chronological holdout is the preferred model for this proof-of-concept dataset. In a production setting, longer historical data, rolling time-series cross-validation, additional business variables, and forecast confidence intervals should be considered.